# 🏗️ Notebook 1: Reddit — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

Reddit-like social news aggregator: subreddits, posts, comments, upvotes, and *ranking*
that surfaces what's interesting right now.

### Functional requirements
- Create **subreddits**; users subscribe.
- Post (link or text), upvote/downvote.
- Threaded comments.
- Ranked feeds: "Hot", "New", "Top", "Rising".

### Non-functional
- Read-heavy (99% reads).
- Hot ranking is **time-sensitive** — it must change every few minutes.
- Comment trees can be deep and wide; load paginated/lazily.

### Back-of-envelope
- 50M DAU, avg 5 page loads → 250M requests/day, ~3k RPS average.
- Peak maybe 10× average → 30k RPS.
- Vote events: perhaps 10M/day. Manageable with one or two Redis clusters.


## Architecture

```
        client
          │
          ▼
    ┌──────────┐
    │ API gtwy │
    └────┬─────┘
         │
   ┌─────┼─────────────┬─────────────┬─────────────┐
   ▼     ▼             ▼             ▼             ▼
  Feed  Post          Vote         Comment       Subreddit
  Svc   Svc           Svc          Svc           Svc
   │     │             │             │             │
   ▼     ▼             ▼             ▼             ▼
 Redis  MySQL        Kafka         MySQL         MySQL
 (hot   (canonical   (votes →      (threaded
  feed) posts)       ranking job)  comments)

   ┌──────────────────────────┐
   │ Ranking job (every 5min) │  reads votes + age
   │  → updates hot scores    │
   └──────────────────────────┘
```

- **Hot feed** is precomputed and cached. Computing it live for every request is unaffordable.
- **Votes** are written to Kafka first (fast), then aggregated. Eventual consistency of counts is fine.
